# **Prophage Detection and Annotation – Pharokka**

## **Tool Information**

- **Tool:**  Pharokka v1.8.2
- **Input:** Assembled genome contigs (FASTA)
- **Organism:** *Acinetobacter baumannii*
- **Analysis type**: Prophage detection and bacteriophage annotation  

Pharokka is a fast and scalable bacteriophage annotation tool designed to identify and annotate prophage regions in bacterial genome assemblies. It integrates multiple sequence similarity approaches and curated phage databases to provide detailed annotation of phage-related genes and genomic regions.

This notebook documents the detection and characterisation of prophage regions from Acinetobacter baumannii genome assemblies using Pharokka.



## **Installation**

### 1) Create a dedicated environment
We create a dedicated conda environment to isolate Pharokka and its dependencies from other tools. This ensures a clean and reproducible setup for phage genome annotation.

In [ ]:
%%bash

conda create -n pharokka_aba python=3.10 -y

### 2) Install Pharokka using mamba
We install Pharokka using pip, as it provides a more reliable installation compared to conda-based methods. This ensures proper installation of the tool and its required Python dependencies.

In [ ]:
%%bash

# Activate environment
source /home/anaconda/miniconda3/etc/profile.d/conda.sh
conda activate pharokka_aba

pip install pharokka

### 3) Verify Installation
We verify that Pharokka is installed correctly by checking the executable path and version. This confirms that the tool is accessible and ready for phage genome annotation analyses.

In [ ]:
%%bash

source /home/anaconda/miniconda3/etc/profile.d/conda.sh
conda activate /home/nidhi/.conda/envs/pharokka_aba

which pharokka.py
pharokka.py --version
pharokka.py --help | head

/home/nidhi/.conda/envs/pharokka/bin/pharokka.py
1.8.2
usage: pharokka.py [-h] [-i INFILE] [-o OUTDIR] [-d DATABASE] [-t THREADS]
                   [-f] [-p PREFIX] [-l LOCUSTAG] [-g GENE_PREDICTOR] [-m]
                   [-s] [-c CODING_TABLE] [-e EVALUE] [--fast]
                   [--mmseqs2_only] [--meta_hmm] [--dnaapler]
                   [--custom_hmm CUSTOM_HMM] [--genbank] [--terminase]
                   [--terminase_strand TERMINASE_STRAND]
                   [--terminase_start TERMINASE_START]
                   [--skip_extra_annotations] [--skip_mash]
                   [--minced_args MINCED_ARGS] [--mash_distance MASH_DISTANCE]
                   [--trna_scan_model {general,bacterial}]


## **Input Files**

The input for Pharokka consists of assembled genome contigs generated by the GHRU Nextflow assembly pipeline.

### Input Requirements

- Genome assemblies in FASTA format  
- One assembly per sample  
- Assemblies should be quality-checked prior to prophage analysis  

## **Database Installation and Initialisation**

Pharokka requires several curated reference databases for accurate prophage
detection and annotation. These databases are not bundled directly with the
software and must be downloaded during the initial setup of the tool.

Database installation was performed once during environment preparation using
the Pharokka database installation utility. This step downloads and configures
the following resources:

- PHROGs protein family database
- VFDB virulence factor database
- CARD antimicrobial resistance database
- INPHARED phage reference sketches
- Mash sketches for phage similarity searches

⚠️ Run the below code only if MOB-suite is being used for the first time

In [ ]:
%%bash

# activate Pharokka environment
source /home/anaconda/miniconda3/etc/profile.d/conda.sh
conda activate pharokka_aba

# move to Pharokka installation directory
cd (path to pharokka.py))

# run database installation script
python install_databases.py

## **Execution Strategy**

Pharokka was executed using a parallelised workflow to efficiently process multiple genome assemblies. Each genome was analysed independently in its own output directory to avoid file conflicts and to ensure traceability of results.

A GNU Parallel based strategy was used to process multiple samples simultaneously, while assigning multiple CPU threads to each individual genome analysis. This hybrid parallelisation approach allowed optimal usage of available computational resources.

For each genome:

- Input FASTA file was provided to Pharokka
- A unique output directory was created
- All annotation steps were performed independently
- Execution status was recorded in a central log file

### Key Parameters

- `-i` → Input genome assembly file  
- `-o` → Output directory for each sample  
- `-t` → Number of CPU threads per analysis  
- `-f` → Force overwrite of existing outputs  
- `-d` → Path to the Pharokka database  

The workflow runs **five Pharokka jobs simultaneously**, each using 10 CPU threads, allowing efficient processing of multiple genome assemblies.

In [ ]:
%%bash

source /home/anaconda/miniconda3/etc/profile.d/conda.sh
conda activate pharokka_aba

# Input assemblies directory
ASSEMBLY_DIR="/data/internship_data/nidhi/aba/new_output/nextflow_output/assemblies"

# Output directory base
OUT_BASE="/data/internship_data/nidhi/aba/new_output/pharokka_output"

# Log file
LOGFILE="/data/internship_data/nidhi/aba/new_output/logs/pharokka.log"

# Pharokka database directory
DB_DIR="/home/nidhi/.conda/envs/pharokka_env/databases/pharokka_db"

# Number of threads for each sample
THREADS=10

# Make output directory if not exists
mkdir -p "$OUT_BASE"
mkdir -p /data/internship_data/nidhi/aba/new_output/logs

# Clear old log
> "$LOGFILE"

# Export variables so GNU parallel can access them
export DB_DIR THREADS OUT_BASE LOGFILE

# Run Pharokka in parallel (5 jobs at once)
ls "$ASSEMBLY_DIR"/*.fasta | parallel -j 5 '
    sample=$(basename {} .fasta)
    outdir="$OUT_BASE/${sample}"

    echo "=== Running Pharokka on $sample ==="

    pharokka.py \
        -i {} \
        -o "$outdir" \
        -t $THREADS \
        -f \
        -d "$DB_DIR"
        >> "$LOGFILE" 2>&1

  ret=$?

    if [ $ret -ne 0 ]; then
        echo "PHAROKKA_FAILED $sample" >> "$LOGFILE"
    else
        echo "PHAROKKA_SUCCESS $sample" >> "$LOGFILE"
    fi

    echo "=== END $sample ===" >> "$LOGFILE"
    '

## **Expected Outputs**

Pharokka generates multiple output files for each genome assembly. The most
important and biologically relevant files used for downstream interpretation
are listed below.

### Annotation Files

- **`pharokka.gb`k**  
  GenBank formatted file containing annotated prophage regions and gene
  annotations. This file is suitable for visualisation in genome browsers.

- **`pharokka.gff`**  
  GFF format file containing coordinates of predicted prophage regions and
  annotated genes.

- **`pharokka.tbl`**  
  Feature table file used for structured annotation reporting.

### Gene and Function Annotation Tables

- **`pharokka_cds_final_merged_output.tsv`**  
  Final merged table containing all predicted coding sequences with
  annotations.

- **`pharokka_cds_functions.tsv`**  
  Functional assignments of predicted prophage genes.

- **`pharokka_length_gc_cds_density.tsv`**  
  Summary statistics including prophage length, GC content and CDS density.

In [31]:
%%bash
ls -1 /data/internship_data/nidhi/aba/new_output/pharokka_output/ABA-1000.short

logs
phanotate.faa
phanotate.ffn
pharokka.gbk
pharokka.gff
pharokka.tbl
pharokka_1768192265.4683983.log
pharokka_aragorn.gff
pharokka_aragorn.txt
pharokka_cds_final_merged_output.tsv
pharokka_cds_functions.tsv
pharokka_length_gc_cds_density.tsv
pharokka_minced.gff
pharokka_minced_spacers.txt
pharokka_top_hits_mash_inphared.tsv
terL.faa
terL.ffn
top_hits_card.tsv
top_hits_vfdb.tsv
trnascan_out.gff
trnascan_out.sec


## **Citation**

Bouras G, Nepal R, Houtak G, Psaltis AJ,
Wormald P-J, Vreugde S.

Pharokka: a fast scalable bacteriophage annotation tool.

Bioinformatics. 2023;39(1):btac776.

https://doi.org/10.1093/bioinformatics/btac776